# Lesson 6-1: MQTT Publisher 測試

本課程將學習如何使用 MQTT 協定發送訊息到 Broker

## 1. 匯入必要的套件

In [1]:
import paho.mqtt.client as mqtt
import time
import json
from datetime import datetime

print("✅ 套件匯入成功！")

✅ 套件匯入成功！


## 2. MQTT 設定

In [2]:
# MQTT Broker 設定
BROKER = "localhost"  # 本地 Raspberry Pi 的 MQTT Broker
PORT = 1883
TOPIC = "客廳/topic"
CLIENT_ID = "pi_publisher_notebook"

print(f"Broker: {BROKER}")
print(f"Port: {PORT}")
print(f"Topic: {TOPIC}")
print(f"Client ID: {CLIENT_ID}")

Broker: localhost
Port: 1883
Topic: 客廳/topic
Client ID: pi_publisher_notebook


## 3. 定義回調函數

In [3]:
# 連線成功的回調函數
def on_connect(client, userdata, flags, rc):
    if rc == 0:
        print("✅ 成功連接到 MQTT Broker！")
    else:
        print(f"❌ 連接失敗，錯誤代碼: {rc}")

# 發布成功的回調函數
def on_publish(client, userdata, mid):
    print(f"📤 訊息已發布 (Message ID: {mid})")

print("✅ 回調函數定義完成")

✅ 回調函數定義完成


## 4. 測試 1：發送單一訊息

In [4]:
# 建立 MQTT 客戶端
client = mqtt.Client(client_id=CLIENT_ID)
client.on_connect = on_connect
client.on_publish = on_publish

# 連接到 Broker
print(f"🔌 正在連接到 {BROKER}:{PORT}...")
client.connect(BROKER, PORT, 60)
client.loop_start()

# 等待連接完成
time.sleep(1)

# 發送單一訊息
message = "Hello from Jupyter Notebook!"
result = client.publish(TOPIC, message, qos=1)

if result.rc == mqtt.MQTT_ERR_SUCCESS:
    print(f"✅ 訊息已送出: {message}")
else:
    print(f"❌ 訊息發送失敗")

time.sleep(1)
client.loop_stop()
client.disconnect()
print("🔌 已斷開連接")

/tmp/ipykernel_55942/1228269539.py:2: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client(client_id=CLIENT_ID)


🔌 正在連接到 localhost:1883...
✅ 成功連接到 MQTT Broker！
✅ 訊息已送出: Hello from Jupyter Notebook!
📤 訊息已發布 (Message ID: 1)
🔌 已斷開連接


## 5. 測試 2：發送 JSON 格式訊息

In [5]:
# 建立新的客戶端
client = mqtt.Client(client_id=CLIENT_ID + "_json")
client.on_connect = on_connect
client.on_publish = on_publish

# 連接
client.connect(BROKER, PORT, 60)
client.loop_start()
time.sleep(1)

# 準備 JSON 訊息
message_data = {
    "時間": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "來源": "Raspberry Pi",
    "溫度": 25.5,
    "濕度": 60,
    "狀態": "正常"
}

message_json = json.dumps(message_data, ensure_ascii=False)
print(f"📝 準備發送 JSON 訊息:")
print(message_json)

# 發送
result = client.publish(TOPIC, message_json, qos=1)

if result.rc == mqtt.MQTT_ERR_SUCCESS:
    print(f"✅ JSON 訊息已送出")
else:
    print(f"❌ 訊息發送失敗")

time.sleep(1)
client.loop_stop()
client.disconnect()
print("🔌 已斷開連接")

/tmp/ipykernel_55942/1362259973.py:2: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client(client_id=CLIENT_ID + "_json")


✅ 成功連接到 MQTT Broker！
📝 準備發送 JSON 訊息:
{"時間": "2025-11-30 11:54:37", "來源": "Raspberry Pi", "溫度": 25.5, "濕度": 60, "狀態": "正常"}
✅ JSON 訊息已送出
📤 訊息已發布 (Message ID: 1)
🔌 已斷開連接


## 6. 測試 3：連續發送多則訊息

In [6]:
# 建立客戶端
client = mqtt.Client(client_id=CLIENT_ID + "_multi")
client.on_connect = on_connect
client.on_publish = on_publish

# 連接
client.connect(BROKER, PORT, 60)
client.loop_start()
time.sleep(1)

# 發送 5 則訊息
print(f"📡 開始發送多則訊息到主題: {TOPIC}")
print("-" * 50)

for i in range(5):
    message_data = {
        "序號": i + 1,
        "時間": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "訊息": f"這是第 {i + 1} 則測試訊息"
    }
    
    message_json = json.dumps(message_data, ensure_ascii=False)
    result = client.publish(TOPIC, message_json, qos=1)
    
    if result.rc == mqtt.MQTT_ERR_SUCCESS:
        print(f"✅ 訊息 {i + 1} 已送出")
    else:
        print(f"❌ 訊息 {i + 1} 發送失敗")
    
    time.sleep(1)  # 每則訊息間隔 1 秒

print("-" * 50)
print("✅ 所有訊息發送完成！")

time.sleep(1)
client.loop_stop()
client.disconnect()
print("🔌 已斷開連接")

/tmp/ipykernel_55942/430508484.py:2: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client(client_id=CLIENT_ID + "_multi")


✅ 成功連接到 MQTT Broker！
📡 開始發送多則訊息到主題: 客廳/topic
--------------------------------------------------
✅ 訊息 1 已送出
📤 訊息已發布 (Message ID: 1)
✅ 訊息 2 已送出
📤 訊息已發布 (Message ID: 2)
✅ 訊息 3 已送出
📤 訊息已發布 (Message ID: 3)
✅ 訊息 4 已送出
📤 訊息已發布 (Message ID: 4)
✅ 訊息 5 已送出
📤 訊息已發布 (Message ID: 5)
--------------------------------------------------
✅ 所有訊息發送完成！
🔌 已斷開連接


## 7. 測試 4：自訂主題發送訊息

In [7]:
# 自訂主題和訊息
custom_topic = "工廠/溫度感測器"  # 可以修改這裡
custom_message = "溫度: 28.5°C"   # 可以修改這裡

# 建立客戶端
client = mqtt.Client(client_id=CLIENT_ID + "_custom")
client.on_connect = on_connect
client.on_publish = on_publish

# 連接並發送
client.connect(BROKER, PORT, 60)
client.loop_start()
time.sleep(1)

print(f"📡 發送到主題: {custom_topic}")
print(f"📝 訊息內容: {custom_message}")

result = client.publish(custom_topic, custom_message, qos=1)

if result.rc == mqtt.MQTT_ERR_SUCCESS:
    print(f"✅ 訊息已成功發送")
else:
    print(f"❌ 訊息發送失敗")

time.sleep(1)
client.loop_stop()
client.disconnect()
print("🔌 已斷開連接")

/tmp/ipykernel_55942/407444079.py:6: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client(client_id=CLIENT_ID + "_custom")


✅ 成功連接到 MQTT Broker！
📡 發送到主題: 工廠/溫度感測器
📝 訊息內容: 溫度: 28.5°C
✅ 訊息已成功發送
📤 訊息已發布 (Message ID: 1)
🔌 已斷開連接
